# Базовый анализ данных и обработка информаций

In [ ]:
import gc
import os
import wordcloud
import pandas as pd
import numpy as np
import os.path as path
import matplotlib.pyplot as plt
from tqdm import tqdm

# Для обработки слов и лементизаций данных
import re
import nltk
import pymorphy3
import string

from nltk.corpus import stopwords # Загружаем стоп слова
from nltk.tokenize import word_tokenize # Загружаем функцию для токенизаций данных
from collections import Counter

# Обработка текста
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Для изменения traceback
from IPython.core.ultratb import AutoFormattedTB
# Для многопоточной обработки данных
from pandarallel import pandarallel

tqdm.pandas()
pandarallel.initialize(progress_bar=True)
nltk.download("stopwords")

# Функция отлова ошибок(взято со StackOverFlow)

In [ ]:
# Инициализация форматтера для преобразования сложных объектов ошибок в простой текст
itb = AutoFormattedTB(mode='Plain', tb_offset=1)

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    """
    Кастомный обработчик исключений для среды IPython/Jupyter.
    
    shell     - Экземпляр InteractiveShell (текущая сессия ноутбука). 
                Позволяет управлять выводом и доступом к ядру.
    etype     - Exception Type: Класс возникшей ошибки (например, FileNotFoundError).
    evalue    - Exception Value: Объект ошибки, содержащий сообщение (текст ошибки).
    tb        - Traceback: Стек вызовов, хранящий информацию о месте возникновения ошибки.
    tb_offset - Смещение стека: Количество верхних уровней вызова, которые нужно 
                скрыть (используется, чтобы не показывать системные механизмы обработки).
    """
    
    # Отображаем стандартное графическое уведомление об ошибке в Jupyter
    # shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)

    # Формируем текстовую версию ошибки (без цветового оформления)
    stb = itb.structured_traceback(etype, evalue, tb)
    sstb = itb.stb2text(stb)

    # Выводим визуальный разделитель для явного уведомления пользователя
    print("\n" + "─"*50)
    print("КРИТИЧЕСКАЯ ОШИБКА: ОБРАБОТКА ПРЕРВАНА\n")
    print(sstb)
    print("─"*50)
    # Здесь можно добавить логирование переменной sstb в файл, если потребуется

# Регистрация функции как глобального обработчика для всех исключений типа Exception
get_ipython().set_custom_exc((Exception,), custom_exc)

In [ ]:
BASEPATH = ".."
RESOURCES = f"{BASEPATH}/Resource"

RUSSIAN_STOP_WORDS = stopwords.words("russian")

extra_stop_words = [
    'также', 'однако', 'который', 'это', 'собственный', 'сообщать', 'заявить', 
    'ранее', 'свой', 'весь', 'мочь', 'стать', 'время', 'год', 'слово', 'новость',
    'отметить', 'рассказать', 'получить', 'являться', 'назвать', 'говорить',
    'частности', 'именно', 'поскольку', 'кроме', 'находиться', 'сообщается'
]
RUSSIAN_STOP_WORDS.extend(extra_stop_words)

# Импорт базы данных

In [ ]:
if (not path.isdir(RESOURCES)):
    raise NotADirectoryError("Путь с Resouces не существует пожалуйста выполните 'git pull' или создайте папку так же загрузите kaddle поддробнее в README.md")
    
if (not path.isfile(f"{RESOURCES}/lenta-ru-news.csv")):
    raise FileNotFoundError("Файл для анализа и обучения моделей не существует пожайлуйста установите файл из kaddle подробнее в README.md")

# Собираем информацию о базе данных

In [ ]:
try:
    if (path.isfile(f"{RESOURCES}/database.parquet")):
        DataBase = pd.read_parquet(f"{RESOURCES}/database.parquet")
    else:
        DataBase = pd.read_csv(f"{RESOURCES}/lenta-ru-news.csv", low_memory=False)
        DataBase.to_parquet(f"{RESOURCES}/database.parquet")
        del DataBase 
        gc.collect()
        DataBase = pd.read_parquet(f"{RESOURCES}/database.parquet")
    if (not DataBase.empty):
        print("─"*50)
        print("База данных инициализирована".center(50))
        print("─"*50)

except Exception as e:
    raise (f"Ошибка при загрузке базы данных: {e}")

In [ ]:
DataBase = DataBase.drop(["url"],axis=1)

In [ ]:
DataBase.head(10)

In [ ]:
DataBase.describe()

In [ ]:
DataBase[['topic']].value_counts().plot(kind='barh',figsize=(15,5), title="Распределение навостей по темам",color='seagreen')
DataBase[['topic']].value_counts()

In [ ]:
DataBase['tags'].value_counts().head(30).plot(
    kind='bar', 
    figsize=(15, 5), 
    title="ТОП-30 тегов по количеству новостей", 
    color='seagreen'
)
DataBase['tags'].value_counts().head(15)

In [ ]:
DataBase['date'] = pd.to_datetime(DataBase['date'])
DataBase

In [ ]:
DataBase.groupby(DataBase['date'].dt.year).size().plot(kind='bar', figsize=(15,5),)

In [ ]:
DataBase['len_chars'] = DataBase['text'].apply(lambda x: len(str(x)))
DataBase['len_words'] = DataBase['text'].apply(lambda x: len(str(x).split()))
# Рассчитаем границы для 99-го перцентиля, чтобы убрать единичные гигантские тексты
limit_chars = DataBase['len_chars'].quantile(0.99)
limit_words = DataBase['len_words'].quantile(0.99)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# График символов (ограничиваем ось X)
ax[0].hist(DataBase['len_chars'], bins=100, range=(0, limit_chars), color='skyblue', edgecolor='black')
ax[0].set_xlim(0, limit_chars) # Ограничиваем до 99% данных
ax[0].set_title('Распределение длины (в символах) - zoomed')
ax[0].set_xlabel('Количество символов')
ax[0].set_ylabel('Частота')

# График слов (ограничиваем ось X)
ax[1].hist(DataBase['len_words'], bins=100, range=(0, limit_words), color='salmon', edgecolor='black')
ax[1].set_xlim(0, limit_words) # Ограничиваем до 99% данных
ax[1].set_title('Распределение длины (в словах) - zoomed')
ax[1].set_xlabel('Количество слов')
ax[1].set_ylabel('Частота')

plt.tight_layout()
plt.show()

# Итог базового иследования базы данных
<div style="padding: 20px; border: 1px solid #238b46; border-radius: 15px; font-family: sans-serif; background-color: white; line-height: 1.5;">
<h2 style="color: #238b46; margin-top: 0; border-bottom: 2px solid #238b46; padding-bottom: 10px;">Анализ структуры данных (800 975 записей)</h2>
<p style="font-size: 1.1em; color: #444;">База представляет собой массив новостей за <b>20 лет</b>. Данные достаточно чистые, но имеют свои особенности:</p>
<table style="width: 100%; border-collapse: collapse; margin-bottom: 20px; color: black;">
<thead>
<tr style="background-color: #238b46; color: white;">
<th style="padding: 10px; border: 1px solid #ddd; text-align: left;">Параметр</th>
<th style="padding: 10px; border: 1px solid #ddd; text-align: left;">Что это значит для нас?</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 10px; border: 1px solid #ddd; background-color: #f9f9f9;"><b>Объем (Count)</b></td>
<td style="padding: 10px; border: 1px solid #ddd;">В базе более <b>800 тысяч</b> статей. Это огромный датасет, подходящий для серьезного машинного обучения.</td>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid #ddd;"><b>Пропуски</b></td>
<td style="padding: 10px; border: 1px solid #ddd;">Почти отсутствуют в заголовках и текстах. Однако около <b>60-70 тысяч</b> новостей не имеют тегов или рубрик.</td>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid #ddd; background-color: #f9f9f9;"><b>Темы (Topic)</b></td>
<td style="padding: 10px; border: 1px solid #ddd;">Всего <b>23 уникальные темы</b>. Самая частая — «Россия» (160к). Это удобная целевая переменная.</td>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid #ddd;"><b>Теги (Tags)</b></td>
<td style="padding: 10px; border: 1px solid #ddd;">Их 94 типа. Самый частый — «Все» (453к), который является «мусорным» тегом-заглушкой.</td>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid #ddd; background-color: #f9f9f9;"><b>Временной охват</b></td>
<td style="padding: 10px; border: 1px solid #ddd;"><b>7393 дня</b> (~20 лет). Пиковая активность зафиксирована в декабре 2019 года.</td>
</tr>
</tbody>
</table>
</div>

# Обработка данных

In [ ]:
morph = pymorphy3.MorphAnalyzer()
cache = {}

def clean_and_lemmatize(text):
    if not isinstance(text, str): return ""
    words = re.findall(r'[а-яёa-z0-9]+', text.lower())
    res = []
    for w in words:
        if w not in RUSSIAN_STOP_WORDS:
            if w not in cache:
                cache[w] = morph.parse(w)[0].normal_form
            lemma = cache[w]
            if lemma not in RUSSIAN_STOP_WORDS:
                res.append(lemma)
    return " ".join(res)

In [ ]:
if path.isfile(f"{RESOURCES}/database_raw.parquet"):
    DataBaseLementaized = pd.read_parquet(f"{RESOURCES}/database_raw.parquet")
else:
    print("Запуск лемматизации... Это займет время.")
    DataBaseLementaized = DataBase.copy() 
    
    DataBaseLementaized['text'] = DataBaseLementaized['text'].parallel_apply(clean_and_lemmatize)
    
    DataBaseLementaized.to_parquet(f"{RESOURCES}/database_raw.parquet")
    
    gc.collect()
DataBaseLementaized

In [ ]:
# 1. Берем выборку, чтобы не повесить компьютер (например, 50 000 записей)
# Если компьютер мощный, можно увеличить до 200 000
sample_text = DataBaseLementaized['text_lemmatized'].sample(n=50000, random_state=42).values

# 2. Соединяем всё в одну гигантскую строку
text_for_cloud = " ".join(sample_text)

# 3. Настройка облака слов
# ВАЖНО: Для русского языка часто нужен шрифт, поддерживающий кириллицу.
# Если вместо букв будут квадраты, укажи путь к шрифту (font_path)
wordcloud = wordcloud.WordCloud(
    width=1200, 
    height=800,
    background_color='white',
    colormap='viridis',
    max_words=200, # Ограничимся 200 самыми частыми словами
    stopwords=RUSSIAN_STOP_WORDS # Используем твой список стоп-слов
).generate(text_for_cloud)

# 4. Визуализация
plt.figure(figsize=(15, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off") # Убираем оси
plt.title("Облако слов (выборка 50к новостей)", fontsize=20)
plt.show()

In [ ]:
topic_counts = DataBaseLementaized['topic'].value_counts(normalize=True) * 100
print("Распределение тем (в %):")
print(topic_counts)

In [ ]:
# 1. Удаляем строки, где нет темы (NaN)
df_clean = DataBaseLementaized.dropna(subset=['topic']).copy()

# 2. Оставляем только те темы, где хотя бы 0.5% данных (отсекаем совсем редкие)
# Это оставит примерно 14-15 основных категорий.
threshold = 0.5 
main_topics = topic_counts[topic_counts > threshold].index
df_clean = df_clean[df_clean['topic'].isin(main_topics)]

print(f"Осталось тем: {len(main_topics)}")
print(f"Размер выборки после чистки: {len(df_clean)}")

# 3. Кодируем темы в числа
le = LabelEncoder()
df_clean['topic_label'] = le.fit_transform(df_clean['topic'])
# Для BERT позже сделаем отдельный маленький sample.
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['text_lemmatized'], 
    df_clean['topic_label'], 
    test_size=0.2, 
    random_state=42,
    stratify=df_clean['topic_label'] 
)

print("Данные разбиты на Train и Test.")